# Preprocessing and data manipulation

**R equivalent:** `subset_samples()`, `prune_taxa()`, `filter_taxa()`,
`tax_glom()`, `rarefy_even_depth()`, `transform_sample_counts()`, `merge_samples()`

All manipulation functions return **new** `Phyloseq` objects; inputs are never
mutated.

In [11]:
import numpy as np

from pyloseq import (
    OtuTable,
    Phyloseq,
    PhyTree,
    SampleData,
    TaxTable,
    filter_taxa,
    psmelt,
    rarefy_even_depth,
    subset_samples,
    tax_glom,
    transform_sample_counts,
)
from pyloseq.datasets.fixtures import load_global_patterns_reference

ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
    tree=PhyTree.from_newick(ref["phy_tree_newick"]),
)
print(gp)

Phyloseq(19216 taxa × 26 samples)
  sample_data: 7 variables
  tax_table:   7 ranks
  phy_tree:    19216 tips


## Subset samples

**R equivalent:** `subset_samples(gp, SampleType %in% c("Feces", "Skin"))`

In [12]:
gp_gut = subset_samples(gp, lambda s: s["SampleType"] in ["Feces", "Skin"])
print(f"After subset: {gp_gut.nsamples} samples (was {gp.nsamples})")
assert gp_gut.nsamples < gp.nsamples

After subset: 7 samples (was 26)


## Filter low-abundance taxa

**R equivalent:** `filter_taxa(gp, function(x) sum(x > 5) > (0.2 * length(x)), TRUE)`

In [13]:
# Keep taxa present with > 5 counts in at least 20% of samples
threshold = 0.2 * gp.nsamples
gp_filtered = filter_taxa(gp, lambda x: (x > 5).sum() > threshold)
print(f"After filter: {gp_filtered.ntaxa} taxa (was {gp.ntaxa})")
assert gp_filtered.ntaxa < gp.ntaxa

After filter: 1971 taxa (was 19216)


## Agglomerate to Family rank

**R equivalent:** `tax_glom(gp, taxrank="Family")`

In [14]:
gp_family = tax_glom(gp, "Family")
print(f"After tax_glom(Family): {gp_family.ntaxa} taxa (was {gp.ntaxa})")
assert gp_family.ntaxa < gp.ntaxa

After tax_glom(Family): 341 taxa (was 19216)


## Rarefy to even depth

**R equivalent:** `rarefy_even_depth(gp, rngseed=42)`

In [15]:
gp_rare = rarefy_even_depth(gp, rng_seed=42)
sample_depths = gp_rare.sample_sums()
print(f"Sample depths after rarefaction: {sample_depths.unique()}")
# All samples should have equal depth
assert sample_depths.nunique() == 1, "Rarefaction should produce equal depths"

Sample depths after rarefaction: [58688.]


## Relative abundance transformation

**R equivalent:** `transform_sample_counts(gp, function(x) x / sum(x))`

In [16]:
gp_rel = transform_sample_counts(gp, lambda x: x / x.sum())
rel_sums = gp_rel.sample_sums()
print(f"Sample sums after relative transform: {rel_sums.round(6).unique()}")
np.testing.assert_allclose(rel_sums.values, 1.0, atol=1e-9)

Sample sums after relative transform: [1.]


## Melt to tidy format

**R equivalent:** `psmelt(gp)`

In [17]:
melted = psmelt(gp_family)
print(melted.shape)
print(melted.head())
assert "OTU" in melted.columns
assert "Abundance" in melted.columns
assert "Sample" in melted.columns

(8866, 17)
      OTU   Sample  Abundance X.SampleID   Primer Final_Barcode  \
0  108964      CL3      809.0        CL3  ILBC_01        AACGCA   
1  108964      CC1       12.0        CC1  ILBC_02        AACTCG   
2  108964      SV1       22.0        SV1  ILBC_03        AACTGT   
3  108964  M31Fcsw        1.0    M31Fcsw  ILBC_04        AAGAGA   
4  108964  M11Fcsw        1.0    M11Fcsw  ILBC_05        AAGCTG   

  Barcode_truncated_plus_T Barcode_full_length SampleType  \
0                   TGCGTT         CTAGCGTGCGT       Soil   
1                   CGAGTT         CATCGACGAGT       Soil   
2                   ACAGTT         GTACGCACAGT       Soil   
3                   TCTCTT         TCGACATCTCT      Feces   
4                   CAGCTT         CGACTGCAGCT      Feces   

                                  Description  Kingdom         Phylum  \
0    Calhoun South Carolina Pine soil, pH 4.9  Archaea  Crenarchaeota   
1    Cedar Creek Minnesota, grassland, pH 6.1  Archaea  Crenarchaeota   
